In [1]:
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow import keras
from sklearn.model_selection import train_test_split
import numpy as np
import matplotlib.pyplot as plt

# 데이터 준비

In [2]:
(x_train, y_train), (x_test, y_test) = imdb.load_data(num_words = 5000)

In [3]:
x_train, x_val, y_train, y_val = train_test_split(x_train, y_train, test_size = 0.2,
                                                  stratify = y_train, random_state = 26)

In [4]:
train_seq = pad_sequences(x_train, maxlen = 100)
val_seq = pad_sequences(x_val, maxlen = 100)
test_seq = pad_sequences(x_test, maxlen = 100)

# 모델 구성

In [6]:
model = keras.Sequential()
model.add(keras.Input(shape = (100,)))
model.add(keras.layers.Embedding(5000, 128))
model.add(keras.layers.Dropout(0.5))
model.add(keras.layers.Conv1D(64, 5, activation = "relu"))
model.add(keras.layers.MaxPool1D(4))
model.add(keras.layers.LSTM(64))
model.add(keras.layers.Dense(1, activation = "sigmoid"))

In [7]:
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)              │ (None, 100, 128)            │         640,000 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_1 (Dropout)                  │ (None, 100, 128)            │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv1d_1 (Conv1D)                    │ (None, 96, 64)              │          41,024 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling1d_1 (MaxPooling1D)       │ (None, 24, 64)              │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ lstm_1 (LSTM)                        │ (None, 64)                  │          33,024 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 1)                   │              65 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 714,113 (2.72 MB)

 Trainable params: 714,113 (2.72 MB)

 Non-trainable params: 0 (0.00 B)

In [8]:
model.compile(loss = "binary_crossentropy", optimizer = "adam", metrics = ["accuracy"])
es_cb = keras.callbacks.EarlyStopping(patience = 4, restore_best_weights = True)
cp_cb = keras.callbacks.ModelCheckpoint("./model/cnnrnn-model.keras", save_best_only = True)

In [9]:
history = model.fit(train_seq, y_train, batch_size = 128, epochs = 100,
                    validation_data = (val_seq, y_val), callbacks = [cp_cb, es_cb])

Epoch 1/100
157/157 ━━━━━━━━━━━━━━━━━━━━ 13s 56ms/step - accuracy: 0.6222 - loss: 0.6119 - val_accuracy: 0.8444 - val_loss: 0.3535
Epoch 2/100
157/157 ━━━━━━━━━━━━━━━━━━━━ 8s 53ms/step - accuracy: 0.8763 - loss: 0.2961 - val_accuracy: 0.8520 - val_loss: 0.3312
Epoch 3/100
157/157 ━━━━━━━━━━━━━━━━━━━━ 9s 57ms/step - accuracy: 0.9060 - loss: 0.2400 - val_accuracy: 0.8584 - val_loss: 0.3306
Epoch 4/100
157/157 ━━━━━━━━━━━━━━━━━━━━ 8s 48ms/step - accuracy: 0.9287 - loss: 0.1943 - val_accuracy: 0.8498 - val_loss: 0.3577
Epoch 5/100
157/157 ━━━━━━━━━━━━━━━━━━━━ 9s 57ms/step - accuracy: 0.9412 - loss: 0.1561 - val_accuracy: 0.8514 - val_loss: 0.3696
Epoch 6/100
157/157 ━━━━━━━━━━━━━━━━━━━━ 8s 49ms/step - accuracy: 0.9573 - loss: 0.1216 - val_accuracy: 0.8514 - val_loss: 0.4434
Epoch 7/100
157/157 ━━━━━━━━━━━━━━━━━━━━ 7s 47ms/step - accuracy: 0.9688 - loss: 0.0911 - val_accuracy: 0.8450 - val_loss: 0.4666


In [10]:
model.evaluate(test_seq, y_test)

782/782 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.8530 - loss: 0.3358


[0.33157768845558167, 0.8533200025558472]